# 08 Inventory Decision Simulation

Forecast accuracy matters because it changes ordering decisions. This notebook applies the same actual demand observations to several forecast-based policies.

The costs below are transparent teaching assumptions—not Walmart costs. Holding cost penalizes leftover units; stockout cost penalizes unmet units.

## 1. Locate the project

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
print("Project root located; Phase 2 modules are importable.")

Project root located; Phase 2 modules are importable.


## 2. Understand the newsvendor rule

For holding cost `Co` and stockout cost `Cu`, the one-period newsvendor critical quantile is `Cu / (Cu + Co)`. When stockouts are four times as expensive as holding, the target quantile is 0.8.

In [2]:
from src.phase2.inventory import newsvendor_quantile

for stockout_cost in [1.0, 4.0, 9.0]:
    quantile = newsvendor_quantile(holding_cost=1.0, stockout_cost=stockout_cost)
    print(f"Stockout cost {stockout_cost:>3}: critical quantile = {quantile:.2f}")

Stockout cost 1.0: critical quantile = 0.50
Stockout cost 4.0: critical quantile = 0.80
Stockout cost 9.0: critical quantile = 0.90


## 3. Simulate policies

We compare median ordering, q90 ordering, midpoint safety stock, and an approximate newsvendor quantity interpolated from the saved q10/q50/q90 values. The interpolation is explicitly approximate because full predictive samples were not saved.

In [3]:
from src.phase2.inventory_experiments import run_inventory_suite

inventory_results = run_inventory_suite(PROJECT_ROOT)
display(inventory_results)

,Model,policy,holding_cost_assumption,stockout_cost_assumption,newsvendor_critical_quantile,mean_order_quantity,overstock_units,understock_units,service_level_percent,total_cost,mean_cost_per_forecast,cost_note
0,DeepAR,median_q50,1.0,1.0,0.5,1.273929,942.0,1646.0,61.461016,2588.0,0.924286,Assumed teaching costs only; not Walmart or re...
1,DeepAR,high_service_q90,1.0,1.0,0.5,3.706429,6462.0,355.0,91.688129,6817.0,2.434643,Assumed teaching costs only; not Walmart or re...
2,DeepAR,midpoint_safety_stock,1.0,1.0,0.5,2.412143,3249.0,766.0,82.065090,4015.0,1.433929,Assumed teaching costs only; not Walmart or re...
3,DeepAR,interpolated_newsvendor_quantile,1.0,1.0,0.5,1.273929,942.0,1646.0,61.461016,2588.0,0.924286,Assumed teaching costs only; not Walmart or re...
4,DeepAR + causal Kalman hybrid,median_q50,1.0,1.0,0.5,1.360000,1105.0,1568.0,63.287286,2673.0,0.954643,Assumed teaching costs only; not Walmart or re...
5,DeepAR + causal Kalman hybrid,high_service_q90,1.0,1.0,0.5,3.607500,6193.0,363.0,91.500819,6556.0,2.341429,Assumed teaching costs only; not Walmart or re...
6,DeepAR + causal Kalman hybrid,midpoint_safety_stock,1.0,1.0,0.5,2.388929,3214.0,796.0,81.362679,4010.0,1.432143,Assumed teaching costs only; not Walmart or re...
7,DeepAR + causal Kalman hybrid,interpolated_newsvendor_quantile,1.0,1.0,0.5,1.360000,1105.0,1568.0,63.287286,2673.0,0.954643,Assumed teaching costs only; not Walmart or re...
8,Phase 1 Kalman Filter,median_q50,1.0,1.0,0.5,1.567500,1504.0,1386.0,67.548583,2890.0,1.032143,Assumed teaching costs only; not Walmart or re...
9,Phase 1 Kalman Filter,high_service_q90,1.0,1.0,0.5,3.778929,6638.0,328.0,92.320300,6966.0,2.487857,Assumed teaching costs only; not Walmart or re...


## 4. Compare policies at the main 1:4 cost ratio

Service level measures the fraction of demanded units fulfilled. Lower total assumed cost is better, but the preferred policy changes when holding and stockout costs change.

In [4]:
main_cost_case = inventory_results[
    inventory_results["stockout_cost_assumption"].eq(4.0)
].sort_values("total_cost")

display(main_cost_case[[
    "Model", "policy", "service_level_percent", "overstock_units",
    "understock_units", "total_cost", "mean_cost_per_forecast",
]])

,Model,policy,service_level_percent,overstock_units,understock_units,total_cost,mean_cost_per_forecast
14,DeepAR,midpoint_safety_stock,82.065090,3249.0,766.0,6313.0,2.254643
18,DeepAR + causal Kalman hybrid,midpoint_safety_stock,81.362679,3214.0,796.0,6398.0,2.285000
22,Phase 1 Kalman Filter,midpoint_safety_stock,85.670803,3980.0,612.0,6428.0,2.295714
19,DeepAR + causal Kalman hybrid,interpolated_newsvendor_quantile,88.878483,5001.0,475.0,6901.0,2.464643
15,DeepAR,interpolated_newsvendor_quantile,89.182861,5177.0,462.0,7025.0,2.508929
20,Phase 1 Kalman Filter,median_q50,67.548583,1504.0,1386.0,7048.0,2.517143
23,Phase 1 Kalman Filter,interpolated_newsvendor_quantile,89.768204,5397.0,437.0,7145.0,2.551786
16,DeepAR + causal Kalman hybrid,median_q50,63.287286,1105.0,1568.0,7377.0,2.634643
12,DeepAR,median_q50,61.461016,942.0,1646.0,7526.0,2.687857
17,DeepAR + causal Kalman hybrid,high_service_q90,91.500819,6193.0,363.0,7645.0,2.730357


## 5. Limitation

This is a one-period simulation: it does not model lead time, replenishment constraints, shelf capacity, spoilage, backorders or multi-echelon inventory. Its purpose is to demonstrate how forecast distributions support decisions beyond point accuracy.